In [16]:
# Improved FLAN-T5 training script (full, Colab-ready)
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    set_seed
)

# -------------------------
# 1) Settings / paths
CSV_PATH = "/content/combined.csv"   # your CSV
MODEL_NAME = "google/flan-t5-base"                 # or "t5-small" for faster runs
OUTPUT_DIR = "./flan_t5_tagline_model_improved"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training hyperparams
SEED = 42
set_seed(SEED)
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 64
MIN_TARGET_WORDS = 3      # filter very-short targets; raise to 5 if you can
NUM_EPOCHS = 6
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
FP16 = False              # set True only after you've confirmed stable fp32 runs
FREEZE_ENCODER = True     # freeze encoder to avoid overfitting small datasets

# Generation hyperparams (for inference)
GEN_MAX_LENGTH = 32
DO_SAMPLE = True
TOP_P = 0.92
TOP_K = 40
TEMPERATURE = 0.9
REPETITION_PENALTY = 1.8
NUM_BEAMS = 4             # used if do_sample=False

# -------------------------
# 2) Load + Clean CSV
# -------------------------

df = pd.read_csv(CSV_PATH, encoding="latin1")





# rename columns if they differ
df = df.rename(columns={
    "Company": "company",
    "Slogans": "slogan",
    "Catagory": "category",
    "description": "description",
}, errors="ignore")


print(f"Rows before cleaning: {len(df)}")



Rows before cleaning: 6942


In [17]:
print(df.loc[0])

company                                                 EgyptAir
slogan                          Makes all the difference. (1978)
Category                                                Airlines
description    Imagine stepping off the plane and onto the su...
Name: 0, dtype: object


In [18]:
# keep only columns we need (safe intersect)
cols = df.columns.intersection(["company", "description", "category", "slogan"])
df = df.loc[:, cols]

# fill / cast / strip
for c in ["company", "description", "category", "slogan"]:
    if c not in df.columns:
        df[c] = ""
    df[c] = df[c].astype(str).fillna("").str.strip()

# drop obviously bad rows
df = df[(df["description"].str.len() > 3) & (df["slogan"].str.len() > 0)].copy()

# filter very short slogans (low signal)
df = df[df["slogan"].str.split().apply(len) >= MIN_TARGET_WORDS].reset_index(drop=True)

print(f"Rows after cleaning: {len(df)}")
if len(df) == 0:
    raise SystemExit("No valid rows after cleaning — relax filters or provide more data.")



Rows after cleaning: 5943


In [19]:
def make_input_text(row):
    parts = []
    # prompt format that helps T5 generalize
    parts.append("task: generate tagline")
    if row.get("company"):
        parts.append(f"Company: {row['company']}")
    if row.get("category"):
        parts.append(f"Category: {row['category']}")
    parts.append(f"Description: {row['description']}")
    return "\n".join(parts)

df["input_text"] = df.apply(make_input_text, axis=1)
df["target_text"] = df["slogan"].astype(str).str.strip()

# Append explicit EOS marker (optional; safe)
# We'll append " </s>" to every target_text to make tokenization consistent
df["target_text"] = df["target_text"].apply(lambda t: t if t.endswith("</s>") else t + " </s>")


# -------------------------
# 2) HF Dataset + split
# -------------------------
hf_ds = Dataset.from_pandas(df[["input_text", "target_text"]])
hf_ds = hf_ds.train_test_split(test_size=0.1, seed=SEED)
test_ds = hf_ds["test"]
# -------------------------
# 3) Tokenizer & Model
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# If pad token missing (rare), set it sensibly
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokenizer PAD:", tokenizer.pad_token_id, "EOS:", tokenizer.eos_token_id)

# Optionally freeze encoder to avoid overfitting small data
if FREEZE_ENCODER:
    for p in model.encoder.parameters():
        p.requires_grad = False
    print("Encoder frozen — training decoder (fewer trainable params).")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Tokenizer PAD: 0 EOS: 1
Encoder frozen — training decoder (fewer trainable params).


In [20]:
# -------------------------
# 4) Preprocessing function
#    - inputs padded to max_length
#    - targets tokenized WITHOUT padding (collator will pad and set -100)
# -------------------------
def preprocess_fn(batch):
    # Tokenize inputs: pad to fixed length
    inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        padding="max_length",
        truncation=True,
    )

    # Tokenize targets WITHOUT padding (so collator will pad later)
    with tokenizer.as_target_tokenizer():
        targets = tokenizer(
            batch["target_text"],
            max_length=MAX_TARGET_LEN,
            padding=False,
            truncation=True,
        )

    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized = hf_ds.map(preprocess_fn, batched=True, remove_columns=hf_ds["train"].column_names)
print(tokenized)

# -------------------------
# 5) Collator: pads labels and converts pad->-100
# -------------------------
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)


# Quick sanity check on a single collated batch
sample_n = min(2, len(tokenized["train"]))
sample_examples = [tokenized["train"][i] for i in range(sample_n)]
batch = collator(sample_examples)
print("Batch keys:", batch.keys())
print("labels shape:", batch["labels"].shape)
print("labels (first row):", batch["labels"][0])
print("valid label tokens (not -100):", (batch["labels"] != -100).sum().item())

# Move model to device and forward a batch to ensure no NaNs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
batch = {k: v.to(device) for k, v in batch.items()}


Map:   0%|          | 0/5348 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/595 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5348
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 595
    })
})
Batch keys: KeysView({'input_ids': tensor([[ 2491,    10,  3806,  7860,   747,  1958,    10, 12886,  7726,    10,
          5264,    24,  1829,    13,  4333,   116,    25,   166,  2525,    12,
          1262,    58, 12886,    65,   118,   740,  3468,  2948,    21,  4160,
             6,   801,    21,   652,    25,   213,    25,   174,    12,   281,
             6,   215,   227,   215,     5,    94,    31,     7,     3,     9,
          3236,  1160,     6,   417,     6,    68,    92,     3,     9, 17090,
            12,   126, 12560,     6,   600,    11,   422,     5,     1,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,

In [21]:
model.train()
out = model(**batch)
print("DEBUG forward loss:", out.loss)
if torch.isnan(out.loss):
    raise RuntimeError("NaN in forward pass — inspect tokenization / targets.")

# -------------------------
# 6) TrainingArguments
# -------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    fp16=FP16,
    report_to="none",
    gradient_accumulation_steps=1,
    label_smoothing_factor=0.0,   # try 0.1 if you see overconfidence
    warmup_steps=0,
    max_grad_norm=1.0,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=collator,
    tokenizer=tokenizer,
)


DEBUG forward loss: tensor(4.3544, device='cuda:0', grad_fn=<NllLossBackward0>)


/tmp/ipython-input-3234254144.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [22]:
# -------------------------
# 9) Train
# -------------------------
trainer.train()

# -------------------------
# 10) Save model + tokenizer
# -------------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Epoch,Training Loss,Validation Loss
1,2.996000,2.742090
2,2.798300,2.675369
3,2.731000,2.646030
4,2.655300,2.628949
5,2.634300,2.619843
6,2.672000,2.617380


('./flan_t5_tagline_model_improved/tokenizer_config.json',
 './flan_t5_tagline_model_improved/special_tokens_map.json',
 './flan_t5_tagline_model_improved/spiece.model',
 './flan_t5_tagline_model_improved/added_tokens.json',
 './flan_t5_tagline_model_improved/tokenizer.json')

In [26]:
# -------------------------
# 9) Generation helper (creative sampling)
# -------------------------
def generate_tagline(company, category, description, max_len=GEN_MAX_LENGTH):
    prompt = (
        "task: generate tagline\n"
        f"Company: {company}\n"
        f"Category: {category}\n"
        f"Description: {description}\n</s>"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    gen = model.generate(
        **inputs,
        max_length=max_len,
        do_sample=DO_SAMPLE,
        top_p=TOP_P,
        top_k=TOP_K,
        temperature=TEMPERATURE,
        repetition_penalty=REPETITION_PENALTY,
        num_return_sequences=3 if DO_SAMPLE else 1,
        early_stopping=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    outputs = [tokenizer.decode(g, skip_special_tokens=True) for g in gen]
    return outputs

# -------------------------
# 10) Quick sample generation
# -------------------------
sample_prompt = {
    "company": "John Burgers",
    "category": "Food",
    "description": "At John Burgers, each burger is crafted with premium, locally sourced ingredients and house-made sauces. With soft toasted buns, perfectly seasoned patties, and creative flavor combinations, it’s a gourmet burger experience you’ll crave again and again."
}
print("Generated taglines:")
print("\n".join(generate_tagline(**sample_prompt)))

sample_prompt = {
    "company": "Whisk It",
    "category": "beverage",
    "description": "Whisk It is a refined whisky crafted from select grains and matured to perfection. Each sip delivers a smooth, full-bodied richness with warm notes of oak, spice, and subtle sweetness. Designed for those who appreciate timeless craftsmanship, Whisk It brings a premium whisky experience that elevates every moment.."
}
print("Generated taglines:")
print("\n".join(generate_tagline(**sample_prompt)))

sample_prompt = {
    "company": "TerraNest Furnishings",
    "category": "household",
    "description": "TerraNest Furnishings specializes in eco-friendly home furniture crafted from recycled wood and sustainable materials. Their catalog includes modular shelves, ergonomic chairs, and minimalist décor made for modern living spaces."
}
print("Generated taglines:")
print("\n".join(generate_tagline(**sample_prompt)))

sample_prompt = {
    "company": "AeroLoom Wearables",
    "category": "sports",
    "description": "AeroLoom Wearables creates minimalist smartwatches and fitness trackers designed for comfort and precision. The brand focuses on accurate health monitoring, long battery life, and seamless integration with mobile devices."
}
print("Generated taglines:")
print("\n".join(generate_tagline(**sample_prompt)))

sample_prompt = {
    "company": "NovaPetra Cosmetics",
    "category": "Cosmetics",
    "description": "NovaPetra Cosmetics is a premium skincare brand that blends botanical extracts with lightweight mineral formulations. Their products focus on hydration, gentle exfoliation, and long-lasting skin nourishment for all skin types."
}
print("Generated taglines:")
print("\n".join(generate_tagline(**sample_prompt)))

sample_prompt = {
    "company": "AstraSky Airlines",
    "category": "Airlines",
    "description": "AstraSky Airlines is a premium international carrier offering long-haul and regional flights with a focus on comfort, reliability, and modern cabin technology. The airline operates a young fleet of fuel-efficient aircraft and provides seamless connectivity across major global hubs. Known for its attentive service and smooth travel experience, AstraSky caters to both business and leisure travelers."
}
print("Generated taglines:")
print("\n".join(generate_tagline(**sample_prompt)))

Generated taglines:
We make it right. 
With great taste you can have a new favorite. 
For burgers you can’t go wrong. 
Generated taglines:
For the true whiskey man. 
Whisk It for the spirit. 
Made to last 
Generated taglines:
The Best of Earth. 
The future of home furniture. 
The only place you'll find a good seat. 
Generated taglines:
The new life. 
Smart watches - Fit for every occasion. 
The best of all time. 
Generated taglines:
Dermatologist-tested. 
The natural beauty of nature 
Beauty for all skin types. 
Generated taglines:
Airline of the Future. 
We Fly the Dream. 
Aircraft to New York. (1983) 


In [24]:
def parse_input_text(text):
    lines = text.split("\n")
    data = {"company": "", "category": "", "description": ""}

    for ln in lines:
        if ln.startswith("Company:"):
            data["company"] = ln.replace("Company:", "").strip()
        elif ln.startswith("Category:"):
            data["category"] = ln.replace("Category:", "").strip()
        elif ln.startswith("Description:"):
            data["description"] = ln.replace("Description:", "").strip()

    return data["company"], data["category"], data["description"]


In [25]:
import csv

OUTPUT_CSV = "test_predictions_combined.csv"

# ---------- parser ----------
def parse_input_text(text):
    lines = text.split("\n")
    data = {"company": "", "category": "", "description": ""}

    for ln in lines:
        if ln.startswith("Company:"):
            data["company"] = ln.replace("Company:", "").strip()
        elif ln.startswith("Category:"):
            data["category"] = ln.replace("Category:", "").strip()
        elif ln.startswith("Description:"):
            data["description"] = ln.replace("Description:", "").strip()

    return data["company"], data["category"], data["description"]


# ---------- CSV writer ----------
rows = []
for i, ex in enumerate(test_ds):
    company, category, description = parse_input_text(ex["input_text"])

    # generate multiple samples (if DO_SAMPLE=True)
    outputs = generate_tagline(company, category, description)

    rows.append({
        "company": company,
        "category": category,
        "description": description,
        "expected_tagline": ex["target_text"],
        "generated_1": outputs[0] if len(outputs) > 0 else "",
        "generated_2": outputs[1] if len(outputs) > 1 else "",
        "generated_3": outputs[2] if len(outputs) > 2 else "",
    })


# ---------- save to CSV ----------
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"\nSaved predictions to: {OUTPUT_CSV}")



Saved predictions to: test_predictions_combined.csv
